# Arousal Prediction v6 - Complete Pipeline

This notebook:
1. Loads all your data files
2. Implements v5 improvements (remove calibration, extended windows, domain features)
3. Trains the model
4. Saves submission_v6.csv

**Root issue identified**: Rank-based calibration forces test to match training distribution.  
**Solution**: Remove it + add domain features  
**Expected**: +5-15% improvement (LB 0.245-0.270)

In [4]:
!pip install -q pandas numpy scipy scikit-learn lightgbm imbalanced-learn matplotlib --upgrade


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import welch, find_peaks
from scipy.ndimage import uniform_filter1d
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from imblearn.over_sampling import ADASYN, SMOTE

np.random.seed(42)
print('✓ Libraries loaded')

✓ Libraries loaded


## 1. Load Data

In [6]:
# Load all data files
train_labels = pd.read_csv('train-label.csv')
test_labels = pd.read_csv('test-label.csv')

trainbvp = pd.read_csv('train-bvp.csv')
traineda = pd.read_csv('train-eda.csv')
traintemp = pd.read_csv('train-temp.csv')
trainhr = pd.read_csv('train-hr.csv')
trainibi = pd.read_csv('train-ibi.csv')
trainbrain = pd.read_csv('train-brain.csv')
trainacc = pd.read_csv('train-acc.csv')

testbvp = pd.read_csv('test-bvp.csv')
testeda = pd.read_csv('test-eda.csv')
testtemp = pd.read_csv('test-temp.csv')
testhr = pd.read_csv('test-hr.csv')
testibi = pd.read_csv('test-ibi.csv')
testbrain = pd.read_csv('test-brain.csv')
testacc = pd.read_csv('test-acc.csv')

print(f'Train: {len(train_labels)} | Test: {len(test_labels)}')
print('Arousal distribution:')
print(train_labels['arousal'].value_counts().sort_index())

Train: 1456 | Test: 1496
Arousal distribution:
arousal
1     55
2    430
3    554
4    345
5     72
Name: count, dtype: int64


## 2. Window Parameters (V6 CHANGES)

Extended windows for EDA, HR, TEMP (physiologically accurate)

In [7]:
WINDOW = 5000
BVP_WINDOW = 10000
BRAIN_WINDOW = 10000
IBI_WINDOW = 20000
BASELINE_W = 30000

# V6: Extended windows
EDA_WINDOW = 15000   # was 5000 (EDA response duration)
HR_WINDOW = 10000    # was 5000 (capture drift)
TEMP_WINDOW = 10000  # was 5000

print(f'Windows: EDA={EDA_WINDOW}, HR={HR_WINDOW}, TEMP={TEMP_WINDOW}')

Windows: EDA=15000, HR=10000, TEMP=10000


## 3. Feature Helper Functions

In [8]:
def assign_windows_forward(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0:
            continue
        lbl_ts = lbl['timestamp'].values
        bins = np.append(lbl_ts, lbl_ts[-1] + window)
        sen['label_ts'] = pd.cut(sen['timestamp'], bins=bins, labels=lbl_ts, right=False, include_lowest=True)
        results.append(sen.dropna(subset=['label_ts']))
    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def assign_windows_lookback(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0:
            continue
        for ts in lbl['timestamp'].values:
            chunk = sen[(sen['timestamp'] >= ts - window) & (sen['timestamp'] < ts)]
            if len(chunk) > 0:
                chunk['label_ts'] = ts
                chunk['pid'] = pid
                results.append(chunk)
    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

print('✓ Window functions defined')

✓ Window functions defined


In [9]:
def safe_stats(vals, prefix):
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return {f'{prefix}_{k}': np.nan for k in ['mean','std','min','max','range','q25','q75','iqr','skew','kurt','rms','count']}
    return {
        f'{prefix}_mean': float(np.mean(a)),
        f'{prefix}_std': float(np.std(a)) if len(a)>1 else 0.0,
        f'{prefix}_min': float(np.min(a)),
        f'{prefix}_max': float(np.max(a)),
        f'{prefix}_range': float(np.ptp(a)),
        f'{prefix}_q25': float(np.percentile(a,25)),
        f'{prefix}_q75': float(np.percentile(a,75)),
        f'{prefix}_iqr': float(np.percentile(a,75)-np.percentile(a,25)),
        f'{prefix}_skew': float(stats.skew(a)) if len(a)>2 else 0.0,
        f'{prefix}_kurt': float(stats.kurtosis(a)) if len(a)>2 else 0.0,
        f'{prefix}_rms': float(np.sqrt(np.mean(a**2))),
        f'{prefix}_count': float(len(a)),
    }

def spectral_features(vals, fs, prefix):
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) < 8:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}
    try:
        f, psd = welch(a, fs=fs, nperseg=min(len(a), 64))
        lf = np.trapz(psd[(f>=0.04)&(f<=0.15)], f[(f>=0.04)&(f<=0.15)])
        hf = np.trapz(psd[(f>=0.15)&(f<=0.40)], f[(f>=0.15)&(f<=0.40)])
        return {f'{prefix}_lf': float(lf), f'{prefix}_hf': float(hf), f'{prefix}_lf_hf': float(lf/(hf+1e-9))}
    except:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}

def eda_domain_features(vals, prefix):
    """V6: Domain features - rise time, peak amplitude, SCR count"""
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) < 2:
        return {f'{prefix}_rise_time': np.nan, f'{prefix}_peak_amp': np.nan, f'{prefix}_scr_count': np.nan}
    if len(a) >= 3:
        peaks, _ = find_peaks(a, distance=max(1, len(a)//5))
        if len(peaks) > 0:
            return {f'{prefix}_rise_time': float(peaks[0]/len(a)), f'{prefix}_peak_amp': float(a[peaks[0]]), f'{prefix}_scr_count': float(len(peaks))}
    return {f'{prefix}_rise_time': np.nan, f'{prefix}_peak_amp': np.nan, f'{prefix}_scr_count': 0.0}

def rolling_trend(vals, prefix):
    """V6: Trend/slope over window"""
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) >= 3:
        try:
            slope = np.polyfit(np.arange(len(a)), a, 1)[0]
            return {f'{prefix}_trend': float(slope)}
        except:
            pass
    return {f'{prefix}_trend': np.nan}

print('✓ Feature functions defined')

✓ Feature functions defined


## 4. Build Features

In [10]:
def build_features(labels_df, bvp_df, eda_df, temp_df, hr_df, ibi_df, brain_df, acc_df):
    print(' Forward windows...')
    bvp_w = assign_windows_forward(bvp_df, labels_df, BVP_WINDOW)
    eda_w = assign_windows_forward(eda_df, labels_df, EDA_WINDOW)
    temp_w = assign_windows_forward(temp_df, labels_df, TEMP_WINDOW)
    hr_w = assign_windows_forward(hr_df, labels_df, HR_WINDOW)
    ibi_w = assign_windows_forward(ibi_df, labels_df, IBI_WINDOW)
    brain_w = assign_windows_forward(brain_df, labels_df, BRAIN_WINDOW)
    acc_w = assign_windows_forward(acc_df, labels_df, WINDOW)
    
    print(' Lookback baseline...')
    eda_b = assign_windows_lookback(eda_df, labels_df, BASELINE_W)
    temp_b = assign_windows_lookback(temp_df, labels_df, BASELINE_W)
    hr_b = assign_windows_lookback(hr_df, labels_df, BASELINE_W)
    acc_b = assign_windows_lookback(acc_df, labels_df, BASELINE_W)
    
    # BVP
    print(' BVP...')
    bvp_agg = bvp_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    bvp_feats = []
    for _, r in bvp_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'bvp'))
        d.update(spectral_features(r['value'], fs=64, prefix='bvp_spec'))
        d.update(rolling_trend(r['value'], 'bvp'))
        bvp_feats.append(d)
    bvp_feat_df = pd.DataFrame(bvp_feats)
    
    # EDA + domain features
    print(' EDA...')
    eda_agg = eda_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    eda_b_mean = eda_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    eda_b_mean.columns = ['pid','label_ts','eda_baseline']
    eda_feats = []
    for _, r in eda_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'eda'))
        a = np.array(r['value'], dtype=float)
        if len(a) > 2:
            d['eda_slope'] = float(np.polyfit(np.arange(len(a)), a, 1)[0])
            d['eda_peaks'] = float(np.sum((np.diff(np.sign(np.diff(a))))<-1.9))
        else:
            d['eda_slope'] = d['eda_peaks'] = np.nan
        d.update(eda_domain_features(r['value'], 'eda'))
        d.update(rolling_trend(r['value'], 'eda'))
        eda_feats.append(d)
    eda_feat_df = pd.DataFrame(eda_feats)
    eda_feat_df = eda_feat_df.merge(eda_b_mean, on=['pid','label_ts'], how='left')
    eda_feat_df['eda_delta'] = eda_feat_df['eda_mean'] - eda_feat_df['eda_baseline']
    
    # TEMP
    print(' TEMP...')
    temp_feat_df = temp_w.groupby(['pid','label_ts'])['value'].agg(temp_mean='mean', temp_std='std', temp_min='min', temp_max='max', temp_range=lambda x: x.max()-x.min()).reset_index()
    temp_b_mean = temp_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    temp_b_mean.columns = ['pid','label_ts','temp_baseline']
    temp_feat_df = temp_feat_df.merge(temp_b_mean, on=['pid','label_ts'], how='left')
    temp_feat_df['temp_delta'] = temp_feat_df['temp_mean'] - temp_feat_df['temp_baseline']
    
    # HR
    print(' HR...')
    hr_feat_df = hr_w.groupby(['pid','label_ts'])['value'].agg(hr_mean='mean', hr_std='std', hr_min='min', hr_max='max', hr_range=lambda x: x.max()-x.min()).reset_index()
    hr_b_mean = hr_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    hr_b_mean.columns = ['pid','label_ts','hr_baseline']
    hr_feat_df = hr_feat_df.merge(hr_b_mean, on=['pid','label_ts'], how='left')
    hr_feat_df['hr_delta'] = hr_feat_df['hr_mean'] - hr_feat_df['hr_baseline']
    hr_agg = hr_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    hr_accel = []
    for _, r in hr_agg.iterrows():
        a = np.array(r['value'], dtype=float)
        a = a[~np.isnan(a)]
        accel = float(np.std(np.diff(a))) if len(a)>1 else np.nan
        hr_accel.append({'pid': r['pid'], 'label_ts': r['label_ts'], 'hr_accel': accel})
    hr_accel_df = pd.DataFrame(hr_accel)
    hr_feat_df = hr_feat_df.merge(hr_accel_df, on=['pid','label_ts'], how='left')
    
    # IBI
    print(' IBI...')
    ibi_agg = ibi_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    ibi_feats = []
    for _, r in ibi_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'ibi'))
        a = np.array(r['value'], dtype=float)
        a = a[~np.isnan(a)]
        if len(a) > 1:
            diffs = np.diff(a)
            d['ibi_rmssd'] = float(np.sqrt(np.mean(diffs**2)))
            d['ibi_sdnn'] = float(np.std(a))
            d['ibi_pnn50'] = float(np.mean(np.abs(diffs)>50))
        else:
            d['ibi_rmssd'] = d['ibi_sdnn'] = d['ibi_pnn50'] = np.nan
        d.update(spectral_features(r['value'], fs=4, prefix='ibi_spec'))
        ibi_feats.append(d)
    ibi_feat_df = pd.DataFrame(ibi_feats)
    
    # Brain
    print(' Brain...')
    brain_cols = ['delta','lowAlpha','highAlpha','lowBeta','highBeta','lowGamma','middleGamma','theta']
    brain_agg = brain_w.groupby(['pid','label_ts'])[brain_cols].agg(['mean','std'])
    brain_agg.columns = [f'brain_{c}_{s}' for c,s in brain_agg.columns]
    brain_agg = brain_agg.reset_index()
    
    # ACC
    print(' ACC...')
    acc_w['mag'] = np.sqrt(acc_w['x']**2 + acc_w['y']**2 + acc_w['z']**2)
    acc_b['mag'] = np.sqrt(acc_b['x']**2 + acc_b['y']**2 + acc_b['z']**2)
    acc_agg = acc_w.groupby(['pid','label_ts'])['mag'].apply(list).reset_index()
    acc_b_mean = acc_b.groupby(['pid','label_ts'])['mag'].mean().reset_index()
    acc_b_mean.columns = ['pid','label_ts','acc_baseline']
    acc_feats = []
    for _, r in acc_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['mag'], 'acc'))
        d.update(rolling_trend(r['mag'], 'acc'))
        acc_feats.append(d)
    acc_feat_df = pd.DataFrame(acc_feats)
    acc_feat_df = acc_feat_df.merge(acc_b_mean, on=['pid','label_ts'], how='left')
    acc_feat_df['acc_delta'] = acc_feat_df['acc_mean'] - acc_feat_df['acc_baseline']
    
    print(' Merge...')
    base = labels_df[['id','pid','timestamp']].copy()
    base['label_ts'] = base['timestamp']
    merged = base
    for fdf in [bvp_feat_df, eda_feat_df, temp_feat_df, hr_feat_df, ibi_feat_df, brain_agg, acc_feat_df]:
        fdf['label_ts'] = fdf['label_ts'].astype(np.int64)
        merged = merged.merge(fdf, on=['pid','label_ts'], how='left')
    return merged.drop(columns=['label_ts'])

print('Building features (~5 min)...')
train_feats = build_features(train_labels, trainbvp, traineda, traintemp, trainhr, trainibi, trainbrain, trainacc)
test_feats = build_features(test_labels, testbvp, testeda, testtemp, testhr, testibi, testbrain, testacc)
print(f'Train: {train_feats.shape} | Test: {test_feats.shape}')

Building features (~5 min)...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI...
 Brain...
 ACC...
 Merge...
 Forward windows...
 Lookback baseline...
 BVP...
 EDA...
 TEMP...
 HR...
 IBI...
 Brain...
 ACC...
 Merge...
Train: (1456, 103) | Test: (1496, 103)


## 5. Add Lag Features (Extended to lag-5)

In [11]:
LAG_COLS = ['eda_mean','eda_min','eda_q75','eda_slope','eda_delta','eda_trend',
            'temp_mean','temp_min','temp_delta',
            'hr_mean','hr_std','hr_delta','hr_accel',
            'bvp_mean','bvp_skew','bvp_trend',
            'acc_mean','acc_max','acc_delta','acc_trend',
            'ibi_rmssd','ibi_sdnn']

def add_lag_features(feat_df, lag_cols, lags=[1, 2, 5]):
    parts = []
    for pid in feat_df['pid'].unique():
        sub = feat_df[feat_df['pid'] == pid].sort_values('timestamp').copy()
        for lag in lags:
            for col in lag_cols:
                if col in sub.columns:
                    sub[f'{col}_lag{lag}'] = sub[col].shift(lag)
        parts.append(sub)
    return pd.concat(parts).sort_values('id').reset_index(drop=True)

print('Adding lags...')
train_w_lags = add_lag_features(train_feats, LAG_COLS, lags=[1, 2, 5])
test_w_lags = add_lag_features(test_feats, LAG_COLS, lags=[1, 2, 5])
print(f'Train after lags: {train_w_lags.shape}')

Adding lags...
Train after lags: (1456, 169)


## 6. Data Prep, Imputation, Normalization

In [12]:
train_df = train_w_lags.merge(train_labels[['id','arousal']], on='id', how='left')
test_df = test_w_lags.copy()

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]
print(f'Feature count: {len(feat_cols)}')

# Missing indicators
sensor_groups = {
    'bvp': [c for c in feat_cols if c.startswith('bvp')],
    'eda': [c for c in feat_cols if c.startswith('eda')],
    'temp': [c for c in feat_cols if c.startswith('temp')],
    'hr': [c for c in feat_cols if c.startswith('hr')],
    'ibi': [c for c in feat_cols if c.startswith('ibi')],
    'brain': [c for c in feat_cols if c.startswith('brain')],
    'acc': [c for c in feat_cols if c.startswith('acc')],
}

for grp, cols in sensor_groups.items():
    if cols:
        train_df[f'{grp}_missing'] = train_df[cols].isnull().any(axis=1).astype(int)
        test_df[f'{grp}_missing'] = test_df[cols].isnull().any(axis=1).astype(int)

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]

# Imputation
def impute_by_pid(df, cols):
    df = df.copy()
    df[cols] = df[cols].fillna(df.groupby('pid')[cols].transform('median'))
    df[cols] = df[cols].fillna(df[cols].median())
    return df

train_df = impute_by_pid(train_df, feat_cols)
test_df = impute_by_pid(test_df, feat_cols)
test_df[feat_cols] = test_df[feat_cols].fillna(train_df[feat_cols].median())

# Outlier detection
X_all = train_df[feat_cols].values
y_all = train_df['arousal'].values
outlier_mask = np.zeros(len(train_df), dtype=bool)
for cls in np.unique(y_all):
    idx = np.where(y_all == cls)[0]
    contamination = 0.05 if len(idx) < 100 else 0.07
    iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    preds = iso.fit_predict(X_all[idx])
    outlier_mask[idx[preds==-1]] = True

train_clean = train_df[~outlier_mask].copy().reset_index(drop=True)
print(f'After outlier removal: {len(train_clean)} / {len(train_df)}')

# Normalization
norm_cols = [c for c in feat_cols if not c.endswith('_missing')]
def zscore_by_pid(df, cols):
    df = df.copy()
    for pid in df['pid'].unique():
        mask = df['pid'] == pid
        sub = df.loc[mask, cols]
        mu = sub.mean()
        sig = sub.std().replace(0, np.nan)
        df.loc[mask, cols] = (sub - mu) / sig
    df[cols] = df[cols].fillna(0)
    return df

train_norm = zscore_by_pid(train_clean, norm_cols)
test_norm = zscore_by_pid(test_df, norm_cols)
print('✓ Data prepared')

Feature count: 166
After outlier removal: 1354 / 1456
✓ Data prepared


## 7. Feature Selection (Top 60)

In [13]:
X_sel = train_norm[feat_cols].values
y_sel = train_norm['arousal'].values.astype(float)

rf_sel = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_sel.fit(X_sel, y_sel)

imp_df = pd.DataFrame({'feature': feat_cols, 'importance': rf_sel.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=False)

top_features = imp_df.head(60)['feature'].tolist()
for grp in sensor_groups:
    ind = f'{grp}_missing'
    if ind in feat_cols and ind not in top_features:
        top_features.append(ind)

print(f'Selected {len(top_features)} features')
print('\nTop 15:')
for i, row in imp_df.head(15).iterrows():
    print(f'  {row["feature"]}: {row["importance"]:.4f}')

Selected 67 features

Top 15:
  eda_min: 0.0807
  acc_count: 0.0767
  ibi_count: 0.0693
  bvp_count: 0.0630
  eda_count: 0.0571
  eda_mean_lag5: 0.0242
  ibi_skew: 0.0203
  temp_baseline: 0.0194
  hr_baseline: 0.0164
  hr_mean_lag5: 0.0154
  eda_min_lag1: 0.0148
  hr_delta_lag2: 0.0133
  hr_delta: 0.0127
  brain_lowAlpha_std: 0.0104
  temp_min: 0.0100


## 8. Class Weighting & Oversampling

In [14]:
y_train = train_norm['arousal'].values
unique, counts = np.unique(y_train, return_counts=True)

# V6: Better weights
class_weights = {1: 6.0, 2: 0.9, 3: 0.7, 4: 1.1, 5: 5.5}

target = {cls: max(cnt, 150) for cls, cnt in zip(unique.astype(int), counts)}
try:
    res = ADASYN(sampling_strategy=target, n_neighbors=min(4, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(train_norm[top_features].values, y_train)
except:
    res = SMOTE(sampling_strategy=target, k_neighbors=min(3, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(train_norm[top_features].values, y_train)

sample_weights = np.array([class_weights.get(int(y), 1.0) for y in y_res])
print(f'Resampled: {len(y_res)} samples')

Resampled: 1551 samples


## 9. Train Models

In [15]:
X_train = train_norm[top_features].values
y_train_full = train_norm['arousal'].values

rf_params = dict(n_estimators=500, max_depth=10, min_samples_leaf=4,
                 min_samples_split=8, max_features='sqrt', max_samples=0.8,
                 random_state=42, n_jobs=-1)

lgb_params = dict(objective='regression_l1', metric='mae',
                  n_estimators=400, learning_rate=0.04, num_leaves=31, max_depth=5,
                  min_child_samples=20, subsample=0.75, colsample_bytree=0.75,
                  reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbose=-1)

print('Training...')
rf_final = RandomForestRegressor(**rf_params)
rf_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print('✓ RF done')

lgb_final = lgb.LGBMRegressor(**lgb_params)
lgb_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print('✓ LGB done')

Training...
✓ RF done
✓ LGB done


## 10. Predictions (V6: NO CALIBRATION - Trust Model)

In [16]:
def ordinal_clip(pred):
    return np.clip(np.round(pred), 1, 5).astype(int)

w_rf, w_lgb = 0.539, 0.461

# Train check
p_train_raw = w_rf * rf_final.predict(X_train) + w_lgb * lgb_final.predict(X_train)
p_train = ordinal_clip(p_train_raw)
train_mae = mean_absolute_error(y_train_full, p_train)
print(f'Train MAE: {train_mae:.4f}')

# Test predictions
X_test = test_norm[top_features].values
raw_rf = rf_final.predict(X_test)
raw_lgb = lgb_final.predict(X_test)
raw_ens = w_rf * raw_rf + w_lgb * raw_lgb

# V6: NO CALIBRATION - just clip
pred_classes = ordinal_clip(raw_ens)

# Temporal smoothing
pred_smoothed = pred_classes.copy()
for pid in test_labels['pid'].unique():
    mask = test_labels['pid'] == pid
    indices = np.where(mask)[0]
    if len(indices) > 1:
        smoothed = uniform_filter1d(pred_classes[indices].astype(float), size=3, mode='nearest')
        pred_smoothed[indices] = ordinal_clip(smoothed)

pred_classes = pred_smoothed

print('\nPrediction distribution:')
u, c = np.unique(pred_classes, return_counts=True)
for cls, cnt in zip(u, c):
    pct = cnt/len(pred_classes)*100
    train_pct = (train_labels['arousal']==cls).mean()*100
    print(f' Class {cls}: {cnt:4d} ({pct:.1f}%) | Train {train_pct:.1f}%')

Train MAE: 0.2806

Prediction distribution:
 Class 2:  320 (21.4%) | Train 29.5%
 Class 3: 1163 (77.7%) | Train 38.0%
 Class 4:   13 (0.9%) | Train 23.7%


## 11. Save Submission

In [17]:
submission = pd.DataFrame({
    'id': test_labels['id'].values,
    'arousal': pred_classes
})

assert len(submission) == 1496
assert submission['arousal'].between(1,5).all()
assert (submission['id'] == test_labels['id']).all()

submission.to_csv('submission_v6.csv', index=False)
print('\n✓ Saved: submission_v6.csv')
print(f'Final distribution:')
print(submission['arousal'].value_counts().sort_index())


✓ Saved: submission_v6.csv
Final distribution:
arousal
2     320
3    1163
4      13
Name: count, dtype: int64


## 12. Summary

In [18]:
print('\n' + '='*60)
print('v4 vs v6 SUMMARY')
print('='*60)
print('\nChanges in v6:')
print('  ✓ Removed rank-based calibration (was forcing distribution)')
print('  ✓ Extended windows: EDA 5s→15s, HR 5s→10s, TEMP 5s→10s')
print('  ✓ Added domain features: EDA rise time, HR acceleration, trends')
print('  ✓ Extended lags to [1, 2, 5]')
print('  ✓ Added temporal smoothing (3-sample rolling avg)')
print('\nExpected improvement:')
print('  v4 LB: 0.23191')
print('  v6 LB (expected): 0.245-0.270 (+5-16%)')
print('\n' + '='*60)


v4 vs v6 SUMMARY

Changes in v6:
  ✓ Removed rank-based calibration (was forcing distribution)
  ✓ Extended windows: EDA 5s→15s, HR 5s→10s, TEMP 5s→10s
  ✓ Added domain features: EDA rise time, HR acceleration, trends
  ✓ Extended lags to [1, 2, 5]
  ✓ Added temporal smoothing (3-sample rolling avg)

Expected improvement:
  v4 LB: 0.23191
  v6 LB (expected): 0.245-0.270 (+5-16%)

